# MarketLens — Exploratory Analysis & Pipeline Walkthrough

This notebook demonstrates the full MarketLens pipeline end-to-end:
1. Data ingestion from Yahoo Finance
2. Validation and cleaning
3. Financial metric calculation
4. Portfolio analytics
5. Forecasting
6. ML regime classification
7. Visualisations

**All metrics are calculated from real historical data. This is not financial advice.**

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.style.use('dark_background')
COLORS = {'blue': '#3b82f6', 'green': '#22c55e', 'red': '#ef4444',
          'amber': '#f59e0b', 'teal': '#14b8a6', 'purple': '#a855f7'}

print('Imports complete.')

## 1. Data Ingestion

In [ ]:
from pipeline.ingest import fetch_market_data
from pipeline.config import ASSET_UNIVERSE

# Fetch a subset for quick demo
demo_tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'JNJ', 'XOM', 'AMZN', 'TSLA']
raw_df = fetch_market_data(tickers=demo_tickers, start='2020-01-01')
print(f'Fetched {len(raw_df):,} rows for {raw_df.ticker.nunique()} tickers')
print(f'Date range: {raw_df.date.min()} to {raw_df.date.max()}')
raw_df.head()

## 2. Validation

In [ ]:
from pipeline.validate import validate_market_data

clean_df, report = validate_market_data(raw_df)
print('Validation Report:')
for k, v in report.items():
    print(f'  {k}: {v}')

## 3. Financial Metrics

In [ ]:
from pipeline.transform import (
    calculate_daily_metrics,
    calculate_full_period_stats,
    calculate_correlation_matrix,
)

enriched_df = calculate_daily_metrics(clean_df)
stats_df    = calculate_full_period_stats(enriched_df)
print('Full-period statistics:')
stats_df[['ticker','sector','annualised_return','annualised_volatility','sharpe_ratio','max_drawdown']].round(4)

## 4. Visualisation — Cumulative Returns

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#090e1a')
ax.set_facecolor('#0f1629')

for ticker, grp in enriched_df.groupby('ticker'):
    grp = grp.sort_values('date')
    ax.plot(pd.to_datetime(grp['date']), grp['cumulative_return'] * 100,
            label=ticker, linewidth=1.2)

ax.axhline(0, color='#4d6080', linestyle='--', linewidth=0.8)
ax.set_title('Cumulative Returns (Adjusted Close)', color='#f0f6ff', pad=15, fontsize=13)
ax.set_ylabel('Return (%)', color='#8fa3c0')
ax.tick_params(colors='#4d6080')
ax.legend(fontsize=8, loc='upper left')
ax.spines[:].set_color('#1e2d4a')
plt.tight_layout()
plt.savefig('../data/cumulative_returns.png', dpi=150, bbox_inches='tight',
            facecolor='#090e1a')
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = calculate_correlation_matrix(enriched_df)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#090e1a')
ax.set_facecolor('#0f1629')

mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(10, 220, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
            annot=True, fmt='.2f', annot_kws={'size': 9},
            ax=ax, linewidths=0.3, linecolor='#0f1629')

ax.set_title('Asset Return Correlation Matrix', color='#f0f6ff', pad=15)
ax.tick_params(colors='#8fa3c0')
plt.tight_layout()
plt.savefig('../data/correlation_matrix.png', dpi=150, bbox_inches='tight',
            facecolor='#090e1a')
plt.show()

## 6. Portfolio Analytics

In [ ]:
from pipeline.portfolio import compute_portfolio_metrics

weights = {'AAPL': 0.20, 'MSFT': 0.20, 'GOOGL': 0.15,
           'JPM':  0.15, 'JNJ':  0.15, 'XOM':   0.10, 'AMZN': 0.05}

sector_map = {t: ASSET_UNIVERSE[t]['sector'] for t in weights}
result = compute_portfolio_metrics(enriched_df, weights, sector_map)

print('Portfolio Analytics Results')
print('=' * 40)
print(f"Ann. Return:     {result['annualised_return']*100:.2f}%")
print(f"Ann. Volatility: {result['annualised_volatility']*100:.2f}%")
print(f"Sharpe Ratio:    {result['sharpe_ratio']:.2f}")
print(f"Max Drawdown:    {result['max_drawdown']*100:.2f}%")
print(f"\nSector Allocation:")
for sector, weight in result['sector_allocation'].items():
    print(f'  {sector}: {weight*100:.1f}%')
print(f"\n{result['disclaimer']}")

## 7. Forecasting

In [ ]:
from models.forecaster import forecast_price_trend

fcast = forecast_price_trend(enriched_df, ticker='AAPL', horizon=30)

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#090e1a')
ax.set_facecolor('#0f1629')

hist_dates = pd.to_datetime(fcast['historical_dates'])
fore_dates = pd.to_datetime(fcast['forecast_dates'])

ax.plot(hist_dates, fcast['historical_prices'], color=COLORS['blue'], lw=1.5, label='Historical')
ax.plot(fore_dates, fcast['forecast_values'],   color=COLORS['teal'], lw=2, ls='--', label='Forecast')
ax.fill_between(fore_dates, fcast['lower_bound'], fcast['upper_bound'],
                alpha=0.15, color=COLORS['teal'], label='90% CI')

ax.axvline(fore_dates[0], color='#4d6080', ls=':', lw=1)
ax.set_title('AAPL — 30-Day Price Trend Estimate (Holt-Winters ETS)',
             color='#f0f6ff', pad=15)
ax.set_ylabel('Adj. Close ($)', color='#8fa3c0')
ax.legend(fontsize=9)
ax.tick_params(colors='#4d6080')
ax.spines[:].set_color('#1e2d4a')

ax.text(0.01, -0.12, fcast['disclaimer'], transform=ax.transAxes,
        fontsize=7, color='#f59e0b', wrap=True)
plt.tight_layout()
plt.savefig('../data/aapl_forecast.png', dpi=150, bbox_inches='tight',
            facecolor='#090e1a')
plt.show()

## 8. Market Regime Classification

In [ ]:
from models.regime_classifier import train_regime_classifier, predict_regimes

# Train
train_result = train_regime_classifier(enriched_df)
print(f"CV Accuracy: {train_result['cv_accuracy']*100:.1f}%")
print(f"\nFeature Importance:")
for feat, imp in sorted(train_result['feature_importance'].items(), key=lambda x: -x[1]):
    bar = '█' * int(imp * 50)
    print(f'  {feat:20s} {bar} {imp*100:.1f}%')

# Predict
pred_df = predict_regimes(enriched_df)
print(f"\nRegime distribution (last 252 days):")
print(pred_df.tail(252)['regime_label'].value_counts())